# tables refit and robustnessRun order matters only within a notebook; each cell below is the code that produced the corresponding table or figure.

In [ ]:
import argparse, os, pathlib, syssys.path.insert(0, str(pathlib.Path.cwd().parent / "mospr"))import paths as _pTABLES = _p.TABLESns = lambda **kw: argparse.Namespace(**kw)

## Refit before/after, gene axisCode: refit_table_figure (this notebook)

In [ ]:
import argparsefrom pathlib import Pathimport numpy as npimport pandas as pdfrom scipy import statsimport matplotlibmatplotlib.use("Agg")import matplotlib.pyplot as pltROOT = _p.ROOTSRC = _p.TABLES / "tables/refit/refit_per_fold_gene.csv"OUT = _p.TABLESimport os as _oimport sys as _sys, pathlib as _pl_sys.path.insert(0, str(_pl.Path(__file__).resolve().parent))import paths as _pif _o.environ.get("MOSPR_WORK_OUT"):    OUT = Path(_o.environ["MOSPR_WORK_OUT"])    SRC = OUT / "tables/refit_per_fold_gene.csv"RENAME = {"AbMIL max-pool": "Max", "AbMIL mean-pool": "Mean", "S4Model": "S4MIL"}ORDER = ["Max", "Mean", "AbMIL", "HE2RNA", "AbReg", "tRNAformer", "ILRA", "S4MIL",         "MambaMIL", "SRMambaMIL", "MOSBY", "SEQUOIA VIS", "2DMamba",         "CPNN", "MoSPR (ours)"]S1, S2 = "stage1_trainval_holdout", "stage2_refit_trainval"def summarise(df, metric):    rows = []    for (c, m), g in df.groupby(["cohort", "model"]):        a = g[g.stage == S1].sort_values("fold")[metric].values        b = g[g.stage == S2].sort_values("fold")[metric].values        if len(a) != 4 or len(b) != 4:            continue        ci = lambda v: stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))        d = b - a        rows.append({"cohort": c, "model": m, "metric": metric,                     "before": a.mean(), "before_ci": ci(a),                     "after": b.mean(), "after_ci": ci(b),                     "delta": d.mean(), "delta_ci": ci(d),                     "folds_up": int((d > 0).sum()),                     "p_paired": stats.ttest_rel(b, a).pvalue,                     **{f"before_f{i}": x for i, x in enumerate(a)},                     **{f"after_f{i}": x for i, x in enumerate(b)}})    r = pd.DataFrame(rows)    r["_o"] = r.model.map({m: i for i, m in enumerate(ORDER)}).fillna(99)    return r.sort_values(["cohort", "_o"]).drop(columns="_o")def to_tex(s, metric, cohorts, path):    models = [m for m in ORDER if m in set(s.model)]    L = [f"% TV refitting before/after - {metric}. stage 1 (fit on train, early stop on val) vs",         "% stage 2 (refit on train+val, lr 1e-4 for 8 epochs). The test set is the same in both.",         r"% Value = 4-fold mean {\tiny$\pm$95\% CI}. Delta is the fold-paired difference and",         r"% the parentheses give how many of the four folds improved.",         r"\begin{tabular}{l" + "ccc" * len(cohorts) + "}", r"\toprule",         "& " + " & ".join(r"\multicolumn{3}{c}{%s}" % c for c in cohorts) + r"\\"]    st = 2    L.append("".join(r"\cmidrule(lr){%d-%d}" % (st + 3 * i, st + 3 * i + 2)                     for i in range(len(cohorts))))    L.append("Method & " + " & ".join(["Before & After & $\\Delta$"] * len(cohorts)) + r"\\")    L.append(r"\midrule")    for m in models:        cells = []        for c in cohorts:            r = s[(s.cohort == c) & (s.model == m)]            if not len(r):                cells += ["--", "--", "--"]                continue            r = r.iloc[0]            cells.append(r"%.3f\,{\tiny$\pm$%.3f}" % (r.before, r.before_ci))            cells.append(r"%.3f\,{\tiny$\pm$%.3f}" % (r.after, r.after_ci))            cells.append(r"\textbf{%+.3f}\,{\tiny(%d/4)}" % (r.delta, r.folds_up))        nm = r"\textbf{MoSPR (ours)}" if m == "MoSPR (ours)" else m        L.append(f"{nm} & " + " & ".join(cells) + r"\\")    L += [r"\bottomrule", r"\end{tabular}", ""]    path.write_text("\n".join(L))def draw(s, metric, cohorts, path):    models = [m for m in ORDER if m in set(s.model)][::-1]    fig, axes = plt.subplots(1, len(cohorts), figsize=(4.1 * len(cohorts), 5.0),                             sharey=True)    axes = np.atleast_1d(axes)    for ax, c in zip(axes, cohorts):        sub = s[s.cohort == c].set_index("model")        for i, m in enumerate(models):            if m not in sub.index:                continue            r = sub.loc[m]            ours = m == "MoSPR (ours)"            ax.plot([r.before, r.after], [i, i], color="#b0b0b0", lw=2, zorder=1)            ax.scatter(r.before, i, s=34, color="#9a9a9a", zorder=3,                       label="Before (train only)" if i == 0 else None)            ax.scatter(r.after, i, s=44, color="#c0392b" if ours else "#2166ac",                       zorder=4, label="After (train+val refit)" if i == 0 else None)        ax.set_yticks(range(len(models)))        ax.set_yticklabels([m.replace(" (ours)", "") for m in models], fontsize=8)        for t, m in zip(ax.get_yticklabels(), models):            if m == "MoSPR (ours)":                t.set_fontweight("bold")        ax.set_title(c, fontsize=11)        ax.set_xlabel(metric, fontsize=9)        ax.grid(axis="x", alpha=0.25, lw=0.5)        for sp in ("top", "right", "left"):            ax.spines[sp].set_visible(False)        ax.tick_params(axis="x", labelsize=8)    axes[0].legend(fontsize=8, frameon=False, loc="lower right")    fig.tight_layout()    fig.savefig(path, dpi=200, bbox_inches="tight")    (path.parent / "pdf").mkdir(exist_ok=True)    fig.savefig(path.parent / "pdf" / (path.stem + ".pdf"), bbox_inches="tight")    plt.close(fig)def main(a):    (OUT / "tables").mkdir(parents=True, exist_ok=True)    (OUT / "figures").mkdir(parents=True, exist_ok=True)    df = pd.read_csv(SRC).replace({"model": RENAME})    df = df[df.cohort.isin(a.cohorts)]    allr = []    for metric in ("SCC", "PCC"):        s = summarise(df, metric)        allr.append(s)        to_tex(s, metric, a.cohorts, OUT / f"tables/refit_before_after_{metric}.tex")        draw(s, f"Gene {metric}", a.cohorts, OUT / f"figures/refit_before_after_{metric}.png")        up = (s.delta > 0).sum()        print(f"[{metric}] {len(s)}  of them up {up}, down {(s.delta < 0).sum()} · "              f"mean delta {s.delta.mean():+.4f}, min delta {s.delta.min():+.4f} "              f"({s.loc[s.delta.idxmin(), 'model']}/{s.loc[s.delta.idxmin(), 'cohort']})")    pd.concat(allr).round(5).to_csv(OUT / "tables/refit_before_after.csv", index=False)    print(f"\nwritten: {OUT}/tables/ · {OUT}/figures/")

In [ ]:
main(ns(cohorts=["BRCA","KIRC","LUAD"]))

## Refit before/after, pathway axisCode: refit_pathway_table_figure (this notebook)

In [ ]:
import argparsefrom pathlib import Pathimport pandas as pdCPNN = Path(str(_p.CODE))class _F62:  passf62 = _F62()f62.summarise, f62.to_tex, f62.draw = summarise, to_tex, drawOUT = Path(str(_p.TABLES))import os as _oimport sys as _sys, pathlib as _pl_sys.path.insert(0, str(_pl.Path(__file__).resolve().parent))import paths as _pif _o.environ.get("MOSPR_WORK_OUT"):    OUT = Path(_o.environ["MOSPR_WORK_OUT"])LABEL = {"hallmark": "Hallmark", "gobp": "GO-BP", "kegg": "KEGG"}def main(a):    (OUT / "tables").mkdir(parents=True, exist_ok=True)    (OUT / "figures").mkdir(parents=True, exist_ok=True)    df = pd.concat([pd.read_csv(OUT / f"tables/refit_pathway_{c}.csv")                    for c in a.cohorts])    keep = []    for coll in a.collections:        col = f"{coll}_scc"        s = f62.summarise(df, col)        keep.append(s)        f62.to_tex(s, f"{LABEL[coll]} SCC (median)", a.cohorts,                   OUT / f"tables/refit_before_after_{coll}.tex")        f62.draw(s, f"{LABEL[coll]} SCC (median)", a.cohorts,                 OUT / f"figures/refit_before_after_{coll}.png")        up, dn = (s.delta > 0).sum(), (s.delta < 0).sum()        print(f"[{coll}] {len(s)} of them up {up}, down {dn}, mean delta {s.delta.mean():+.4f}")        for r in s[s.delta < 0].itertuples():            print(f"    down {r.model}/{r.cohort} {r.delta:+.4f} "                  f"(folds improved {r.folds_up}/4, p={r.p_paired:.2f})")    pd.concat(keep).round(5).to_csv(OUT / "tables/refit_before_after_pathway.csv",                                    index=False)    print(f"\nwritten: {OUT}/tables/refit_before_after_{{hallmark,gobp,kegg}}.tex · "          f"refit_before_after_pathway.csv\n      {OUT}/figures/ (png + pdf/)")

In [ ]:
main(ns(cohorts=["BRCA","KIRC","LUAD"], collections=["hallmark","gobp","kegg"]))

## Scorer agreement: mean-z vs ssGSEA vs GSVACode: robustness_table (this notebook)

In [ ]:
import argparsefrom pathlib import Pathimport numpy as npimport pandas as pdfrom scipy import statsOUT = Path(str(_p.TABLES))import os as _oimport sys as _sys, pathlib as _pl_sys.path.insert(0, str(_pl.Path(__file__).resolve().parent))import paths as _pif _o.environ.get("MOSPR_WORK_OUT"):    OUT = Path(_o.environ["MOSPR_WORK_OUT"])T = OUT / "tables"ORDER = ["Max", "Mean", "AbMIL", "HE2RNA", "AbReg", "tRNAformer", "ILRA", "S4MIL",         "MambaMIL", "SRMambaMIL", "MOSBY", "SEQUOIA VIS", "2DMamba",         "CPNN", "MoSPR (ours)"]SCORERS = [("meanz", "mean-z"), ("ssgsea", "ssGSEA"), ("gsva", "GSVA")]def load(cohort, coll):    mz = (pd.read_csv(T / f"refit_pathway_{cohort}.csv")          .query("stage=='stage2_refit_trainval'")[["model", "fold", f"{coll}_scc"]]          .rename(columns={f"{coll}_scc": "scc"}).assign(method="meanz"))    tag = "" if coll == "hallmark" else f"_{coll}"    rb = pd.read_csv(T / f"pathway_robustness_{cohort}{tag}.csv")[["model", "fold", "method", "scc"]]    d = pd.concat([mz, rb]); d["cohort"] = cohort    return ddef ci(v):    v = np.asarray(v, float)    return stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))def main(a):    df = pd.concat([load(c, a.collection) for c in a.cohorts])    g = (df.groupby(["cohort", "method", "model"]).scc         .agg(mean="mean", ci=ci, n="size").reset_index())    g = g[g.n == 4]    g.round(5).to_csv(T / f"robustness_summary_{a.collection}.csv", index=False)                      print(f"\n===== {a.collection} — MoSPR rank per scorer / winner =====")    for c in a.cohorts:        line = []        for k, lab in SCORERS:            s = g[(g.cohort == c) & (g.method == k)].set_index("model")["mean"]            if not len(s):                line.append(f"{lab}: --"); continue            r = int(s.rank(ascending=False)["MoSPR (ours)"])            line.append(f"{lab}: MoSPR rank {r}, winner {s.idxmax()}")        print(f"  {c}  " + " | ".join(line))    L = [f"% Pathway-metric robustness: predictions fixed, only the scorer changes ({a.collection}).",         r"% mean-z = the metric used in the paper; ssGSEA/GSVA = official gseapy 1.3.1 implementations.",         r"% Value = 4-fold mean {\tiny$\pm$95\% CI} of the per-set Spearman median.",         r"\begin{tabular}{l" + "ccc" * len(a.cohorts) + "}", r"\toprule",         "& " + " & ".join(r"\multicolumn{3}{c}{%s}" % c for c in a.cohorts) + r"\\",         "".join(r"\cmidrule(lr){%d-%d}" % (2 + 3 * i, 4 + 3 * i)                 for i in range(len(a.cohorts))),         "Method & " + " & ".join([" & ".join(l for _, l in SCORERS)] * len(a.cohorts)) + r"\\",         r"\midrule"]    best = {(c, k): g[(g.cohort == c) & (g.method == k)]["mean"].max()            for c in a.cohorts for k, _ in SCORERS}    for m in [x for x in ORDER if x in set(g.model)]:        cells = []        for c in a.cohorts:            for k, _ in SCORERS:                r = g[(g.cohort == c) & (g.method == k) & (g.model == m)]                if not len(r):                    cells.append("--"); continue                r = r.iloc[0]                s = r"%.3f\,{\tiny$\pm$%.3f}" % (r["mean"], r["ci"])                cells.append(r"\textbf{%s}" % s if np.isclose(r["mean"], best[(c, k)]) else s)        nm = r"\textbf{MoSPR (ours)}" if m == "MoSPR (ours)" else m        L.append(f"{nm} & " + " & ".join(cells) + r"\\")    L += [r"\bottomrule", r"\end{tabular}", ""]    (T / f"appendix_robustness_{a.collection}.tex").write_text("\n".join(L))                         gf = (pd.read_csv(T / "pathway_global_factor.csv")          .groupby(["cohort", "model"])[["gfac_scc", "gfac_share_of_setscore_R2"]].mean()          .reset_index())    P = gf.pivot(index="model", columns="cohort", values="gfac_share_of_setscore_R2")    Q = gf.pivot(index="model", columns="cohort", values="gfac_scc")    L2 = [r"% Share of the mean-z set score explained by a transcriptome-wide slide factor, and",          r"% that factor's own accuracy. A larger share makes scorers disagree more.",          r"\begin{tabular}{l" + "cc" * len(a.cohorts) + "}", r"\toprule",          "& " + " & ".join(r"\multicolumn{2}{c}{%s}" % c for c in a.cohorts) + r"\\",          "".join(r"\cmidrule(lr){%d-%d}" % (2 + 2 * i, 3 + 2 * i) for i in range(len(a.cohorts))),          "Method & " + " & ".join([r"share & $\rho$"] * len(a.cohorts)) + r"\\", r"\midrule"]    for m in [x for x in ORDER if x in P.index]:        cells = []        for c in a.cohorts:            cells += ["%.2f" % P.loc[m, c], "%.3f" % Q.loc[m, c]]        nm = r"\textbf{MoSPR (ours)}" if m == "MoSPR (ours)" else m        L2.append(f"{nm} & " + " & ".join(cells) + r"\\")    L2 += [r"\bottomrule", r"\end{tabular}", ""]    (T / "appendix_global_factor.tex").write_text("\n".join(L2))    print(f"\nwritten: {T}/appendix_robustness_{a.collection}.tex · "          f"appendix_global_factor.tex · robustness_summary_{a.collection}.csv")

In [ ]:
main(ns(cohorts=["BRCA","KIRC","LUAD"], collection="hallmark"))